# Regression Analysis: A Hands-On Introduction

**What you will learn**
1. What regression is and the end-to-end process
2. The maths: solving the simple and multiple linear regression equations by hand
3. Real-world examples and applications
4. Simple Linear Regression with scikit-learn
5. Multiple Linear Regression with scikit-learn
6. The standard modelling steps, evaluation metrics and diagnostics
7. Practice questions

> Run the cells top to bottom. Everything uses synthetic data, so no downloads are needed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)
plt.rcParams["figure.figsize"] = (6, 4)

## 1. What is Regression Analysis?

Regression models the relationship between a **target** (dependent variable, `y`) and one or more **features** (independent variables, `X`) so we can **explain** the relationship and **predict** `y` for new data.

| Type | # features | Equation |
|---|---|---|
| Simple linear regression | 1 | $y = \beta_0 + \beta_1 x + \varepsilon$ |
| Multiple linear regression | 2+ | $y = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p + \varepsilon$ |

- $\beta_0$ = **intercept** (value of y when all x = 0)
- $\beta_1$ = **slope** (change in y for a one-unit increase in x, others held constant)
- $\varepsilon$ = **error** (what the model cannot explain)

The target must be **continuous** (price, temperature, sales). For categories (yes/no) use classification instead.

### The Regression Process at a Glance
```
Define problem -> Collect data -> Explore (EDA) -> Clean & prepare
      -> Split train/test -> Train model -> Evaluate -> Check assumptions
      -> Improve -> Deploy & monitor
```

## 2. The Maths: Solving the Equation

### 2.1 Ordinary Least Squares (OLS)
We choose $\beta_0, \beta_1$ that **minimise the sum of squared errors (SSE)**:

$$SSE = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = \sum_{i=1}^{n} (y_i - \beta_0 - \beta_1 x_i)^2$$

Setting the derivatives to zero gives the closed-form solution:

$$\beta_1 = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2} \qquad\qquad \beta_0 = \bar{y} - \beta_1 \bar{x}$$

### 2.2 Worked Example (by hand)
Hours studied (x) vs exam score (y) for 5 students:

| Student | x (hours) | y (score) |
|---|---|---|
| 1 | 1 | 50 |
| 2 | 2 | 55 |
| 3 | 3 | 65 |
| 4 | 4 | 70 |
| 5 | 5 | 85 |

- $\bar{x} = 3$, $\bar{y} = 65$
- $\sum (x-\bar{x})(y-\bar{y}) = (-2)(-15)+(-1)(-10)+(0)(0)+(1)(5)+(2)(20) = 30+10+0+5+40 = 85$
- $\sum (x-\bar{x})^2 = 4+1+0+1+4 = 10$
- $\beta_1 = 85/10 = 8.5$
- $\beta_0 = 65 - 8.5 \times 3 = 39.5$

**Fitted line:** $\hat{y} = 39.5 + 8.5x$  → each extra study hour adds about 8.5 marks.
Prediction for 6 hours: $39.5 + 8.5 \times 6 = 90.5$.

In [ ]:
# Verify the hand calculation with code
x = np.array([1, 2, 3, 4, 5])
y = np.array([50, 55, 65, 70, 85])

b1 = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean())**2)
b0 = y.mean() - b1 * x.mean()
print(f"Slope (b1)     = {b1}")
print(f"Intercept (b0) = {b0}")
print(f"Prediction for 6 hours = {b0 + b1*6}")

### 2.3 Multiple Regression in Matrix Form (Normal Equation)
With $p$ features, write the model as $\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\varepsilon}$, where $\mathbf{X}$ has a leading column of 1s for the intercept. Minimising SSE gives the **normal equation**:

$$\boldsymbol{\hat\beta} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

In [ ]:
# Normal equation on the same data
X = np.column_stack([np.ones(len(x)), x])        # add intercept column
beta = np.linalg.inv(X.T @ X) @ X.T @ y
print("Normal equation result [b0, b1]:", beta)

## 3. Examples and Applications

| Domain | Target (y) | Typical features (X) |
|---|---|---|
| Real estate | House price | Area, bedrooms, age, location |
| Marketing | Sales / revenue | TV, radio, online ad spend |
| Finance | Stock return / credit risk | Interest rate, volume, income |
| Healthcare | Blood pressure, length of stay | Age, BMI, medication dose |
| Agriculture | Crop yield | Rainfall, fertiliser, temperature |
| Energy | Electricity demand | Temperature, hour of day, holidays |
| HR | Salary | Experience, education, role |
| Operations | Delivery time | Distance, traffic, order size |

**Two main uses:** *Prediction* (forecast new values) and *Inference* (understand which variables matter and by how much).

## 4. Simple Linear Regression with scikit-learn

**Scenario:** predict a student's exam score from hours studied. We generate 100 noisy observations around a true line.

In [ ]:
n = 100
hours = np.random.uniform(0, 10, n)
score = 40 + 5 * hours + np.random.normal(0, 5, n)   # true: intercept 40, slope 5, noise sd 5

df_simple = pd.DataFrame({"hours": hours, "score": score})
df_simple.head()

In [ ]:
# Step 1: Explore
print(df_simple.describe().round(2))
print("\nCorrelation:", round(df_simple["hours"].corr(df_simple["score"]), 3))

plt.scatter(df_simple.hours, df_simple.score, alpha=0.7)
plt.xlabel("Hours studied"); plt.ylabel("Exam score"); plt.title("Hours vs Score")
plt.show()

In [ ]:
# Step 2: Split (X must be 2-D in scikit-learn)
X = df_simple[["hours"]]
y = df_simple["score"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Train
model = LinearRegression()
model.fit(X_train, y_train)

print(f"Intercept: {model.intercept_:.2f}")
print(f"Slope:     {model.coef_[0]:.2f}")
print("Equation:  score = {:.2f} + {:.2f} * hours".format(model.intercept_, model.coef_[0]))

In [ ]:
# Step 4: Predict & evaluate
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
print(f"MAE  = {mae:.2f}\nRMSE = {rmse:.2f}\nR2   = {r2:.3f}")

# Step 5: Visualise the fit
plt.scatter(X_test, y_test, label="Actual", alpha=0.7)
xs = np.linspace(0, 10, 100).reshape(-1, 1)
plt.plot(xs, model.predict(pd.DataFrame(xs, columns=["hours"])), color="red", label="Fitted line")
plt.xlabel("Hours studied"); plt.ylabel("Score"); plt.legend(); plt.show()

# Predict for a new student
print("Predicted score for 7.5 hours:",
      round(model.predict(pd.DataFrame({"hours": [7.5]}))[0], 1))

## 5. Multiple Linear Regression with scikit-learn

**Scenario:** predict house price (in $1000s) from size, bedrooms, age and distance to city centre.

In [ ]:
n = 500
area     = np.random.normal(1500, 400, n)          # sq ft
bedrooms = np.random.randint(1, 6, n)
age      = np.random.uniform(0, 40, n)             # years
distance = np.random.uniform(1, 30, n)             # km

price = (50 + 0.12 * area + 10 * bedrooms - 1.5 * age - 3 * distance
         + np.random.normal(0, 20, n))

df = pd.DataFrame({"area": area, "bedrooms": bedrooms, "age": age,
                   "distance": distance, "price": price})
df.head()

In [ ]:
# EDA: correlations (also look for multicollinearity between features)
corr = df.corr().round(2)
print(corr)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
plt.colorbar(im); plt.title("Correlation matrix"); plt.show()

In [ ]:
# Split, scale (optional for OLS but useful to compare coefficients), train
features = ["area", "bedrooms", "age", "distance"]
X = df[features]; y = df["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

mlr = LinearRegression().fit(X_train, y_train)

coefs = pd.DataFrame({"feature": features, "coefficient": mlr.coef_})
print("Intercept:", round(mlr.intercept_, 2))
print(coefs.round(3))

**Reading the coefficients:** e.g. the `area` coefficient is the expected change in price for **one extra sq ft, holding the other features constant**. A negative coefficient (age, distance) means price falls as that feature rises.

In [ ]:
# Evaluate
y_pred = mlr.predict(X_test)
r2 = r2_score(y_test, y_pred)
n_, p_ = X_test.shape
adj_r2 = 1 - (1 - r2) * (n_ - 1) / (n_ - p_ - 1)

print(f"MAE         = {mean_absolute_error(y_test, y_pred):.2f}")
print(f"RMSE        = {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"R2          = {r2:.3f}")
print(f"Adjusted R2 = {adj_r2:.3f}")

# Cross-validation gives a more reliable estimate than one split
cv = cross_val_score(LinearRegression(), X, y, cv=5, scoring="r2")
print("5-fold CV R2:", cv.round(3), "| mean =", cv.mean().round(3))

In [ ]:
# Standardised coefficients: compare feature importance on a common scale
pipe = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
std_coefs = pd.Series(pipe.named_steps["linearregression"].coef_, index=features)
std_coefs.sort_values().plot(kind="barh", title="Standardised coefficients")
plt.show()

In [ ]:
# Residual diagnostics
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(y_pred, residuals, alpha=0.6)
axes[0].axhline(0, color="red"); axes[0].set_title("Residuals vs Predicted")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Residual")

axes[1].hist(residuals, bins=20); axes[1].set_title("Residual distribution")

axes[2].scatter(y_test, y_pred, alpha=0.6)
lims = [y_test.min(), y_test.max()]
axes[2].plot(lims, lims, color="red"); axes[2].set_title("Actual vs Predicted")
axes[2].set_xlabel("Actual"); axes[2].set_ylabel("Predicted")
plt.tight_layout(); plt.show()

In [ ]:
# Predict for a new house
new_house = pd.DataFrame({"area": [1800], "bedrooms": [3], "age": [10], "distance": [8]})
print(f"Predicted price: ${mlr.predict(new_house)[0]:.0f}k")

## 6. The Modelling Steps (Checklist)

| # | Step | What to do |
|---|---|---|
| 1 | **Define the problem** | Target, goal (predict vs explain), success metric |
| 2 | **Collect data** | Sufficient, relevant, representative rows |
| 3 | **Explore (EDA)** | Distributions, scatter plots, correlations, outliers |
| 4 | **Clean & prepare** | Handle missing values, duplicates, outliers; encode categoricals (one-hot); create features |
| 5 | **Split data** | Train/test (e.g. 80/20) or cross-validation. Never fit on the test set |
| 6 | **Scale (if needed)** | Standardise for regularised models or to compare coefficients. Fit scaler on train only |
| 7 | **Train** | `model.fit(X_train, y_train)` |
| 8 | **Evaluate** | MAE, RMSE, R², Adjusted R² on the **test** set |
| 9 | **Check assumptions** | Residual plots, multicollinearity |
| 10 | **Improve** | Add/remove features, transform variables, polynomial terms, Ridge/Lasso |
| 11 | **Deploy & monitor** | Save the model (`joblib`), track performance drift |

### Evaluation Metrics
- **MAE** = mean of |error|: average miss, in target units
- **MSE / RMSE**: penalises large errors more; RMSE is in target units
- **R²** = 1 - SSE/SST: share of variance explained (0 to 1 for sensible models)
- **Adjusted R²**: R² penalised for extra features; use it when comparing models with different numbers of features

### Key Assumptions of Linear Regression (LINE)
1. **L**inearity: relationship between X and y is linear
2. **I**ndependence: errors are independent of each other
3. **N**ormality: residuals are roughly normal (matters mainly for inference)
4. **E**qual variance (homoscedasticity): residual spread is constant
5. Also: **no severe multicollinearity** between features

### Common Pitfalls
- Extrapolating far outside the training range
- Confusing correlation with causation
- Overfitting with too many features (watch the train vs test gap)
- Ignoring outliers that distort the line
- Data leakage (scaling or selecting features before the split)

## 7. Questions for Participants

### A. Concept check
1. What is the difference between a dependent and an independent variable? Give one example of each.
2. What does the intercept mean in the study-hours model? Is it always meaningful?
3. In multiple regression, how do you interpret a coefficient of -3 on `distance`?
4. Why do we minimise the *squared* errors rather than the plain sum of errors?
5. Explain the difference between R² and Adjusted R². When would they differ a lot?
6. Why do we evaluate on a test set instead of the training set?
7. Name the four LINE assumptions. How would you check each one?

### B. By-hand calculation
8. For the data below, compute the slope and intercept by hand and predict y when x = 6.

   | x | 2 | 4 | 6 | 8 |
   |---|---|---|---|---|
   | y | 3 | 7 | 5 | 10 |

### C. Coding exercises (use the cells above)
9. Change the noise level in the simple regression data from `5` to `20`. What happens to R² and RMSE? Why?
10. Remove the `distance` feature from the house model. How do R² and Adjusted R² change?
11. Add a new feature `rooms_per_area = bedrooms / area` to the house data. Does it help?
12. Add a highly correlated copy of `area` (e.g. `area * 0.093 + noise`, which is square metres). What happens to the coefficients? Which problem is this?
13. Add one extreme outlier to the simple-regression data. How much does the slope change?
14. Replace `LinearRegression` with `Ridge` and `Lasso`. Compare coefficients and CV scores.
15. Train a model on `hours` in the range 0-5 only and predict for 10 hours. What goes wrong and why?

### D. Discussion
16. A model has R² = 0.95 on training data and 0.60 on test data. What is happening and how would you fix it?
17. Your residual plot shows a funnel shape (spread grows with predicted value). Which assumption is violated and what could you try (hint: log-transform y)?
18. Pick a problem from your own work. What would be the target, what features would you use, and how would you tell whether the model is good enough?

---
*End of notebook. Next topics: polynomial regression, regularisation (Ridge/Lasso), logistic regression.*